In [55]:
import bibtexparser
import feedparser
import os
import re
import requests
import pandas as pd
import numpy as np
import time
import urllib.request as libreq

In [114]:
from dotenv import load_dotenv
load_dotenv()

SCOPUS_API_KEY = os.getenv("SCOPUS_API_KEY")  # Required for Scopus API access

DATE_STAMP = "2026-11-10"  # Date stamp for the output files
ACM_FILE_NAME = f"../data/acm-literature-{DATE_STAMP}.bib"  # acm-collection = Full-Text Collection, acm-literature = Guide to Computing Literature
IEEE_FILE_NAME = f"../data/ieee-{DATE_STAMP}.csv"
SPRINGER_FILE_NAME = f"../data/springer-{DATE_STAMP}.csv"

Search strings used for web-based search in ACM, IEEE, and Springer.

**ACM:**
("language model*" OR llm*) AND ("knowledge graph*" OR kg*) AND (hallucinat*)

**IEEE:**
("Abstract":"language model*" OR llm*) AND ("Abstract":"knowledge graph*" OR kg) AND ("Abstract":hallucinat*)
\# Does not allow wildcards for 2 or less characters

**Springer:**
("language model*" OR llm* OR encoder* OR decoder* OR gpt OR rag) AND (knowledge* OR graph* OR kg*) AND (hallucinat* OR fact* OR qa OR trust* OR coheren* OR reliab* OR explain*)

In [57]:
dblp_query = (
    "llm|language|encoder|decoder|gpt|rag "
    "kg|knowledge|graph "
    "hallucinat|fact|qa|trust|coheren|reliab|explain"
)
acl_llm_keywords = "llm|language model"
acl_kg_keywords = "knowledge graph|kg"  # KG due to string.contains may retrieve more false positives (e.g., bacKGround)
acl_hallucination_keywords = "hallucinat"

scopus_query = 'TITLE-ABS-KEY("language model*" OR llm* AND "knowledge graph*" OR kg* AND hallucinat*)'

### DBLP

In [58]:
dblp_df = pd.read_parquet("../data/dblp_anthology.parquet")
print(f"{len(dblp_df):,} DBLP papers loaded")

2,713,161 DBLP papers loaded


In [59]:
def compile_dblp_query(query: str) -> list[str]:
    """Space-separated groups are AND-ed; '|' terms within a group are OR-ed.
    Each term matches the start of a word, like DBLP's prefix search."""
    patterns = []
    for group in query.split():
        terms = [re.escape(t) for t in group.split("|") if t]
        patterns.append(r"\b(?:" + "|".join(terms) + ")")
    return patterns

QUERY_PATTERNS = compile_dblp_query(dblp_query)
QUERY_PATTERNS

['\\b(?:llm|language|encoder|decoder|gpt|rag)',
 '\\b(?:kg|knowledge|graph)',
 '\\b(?:hallucinat|fact|qa|trust|coheren|reliab|explain)']

In [60]:
def search_dblp_local(df: pd.DataFrame, patterns: list[str], col: str = "title") -> pd.DataFrame:
    mask = pd.Series(True, index=df.index)
    for p in patterns:
        mask &= df[col].str.contains(p, case=False, regex=True, na=False)
    return df[mask].copy()

dblp_results = search_dblp_local(dblp_df, QUERY_PATTERNS)
print(f"{len(dblp_results):,} matching papers")

356 matching papers


In [61]:
print(dblp_results["year"].value_counts().sort_index())

year
2020      4
2021      9
2022     11
2023     29
2024     51
2025    131
2026    121
Name: count, dtype: int64


In [62]:
print(dblp_results.iloc[0].to_string())

dblp_key                                 conf/sigir/Shami0S25
title        Fact Verification in Knowledge Graphs Using LLMs
authors     [Farzad Shami, Stefano Marchesin 0001, Gianmar...
year                                                     2025
venue                                                   SIGIR
type                           Conference and Workshop Papers
doi                                   10.1145/3726302.3730142
url                 https://dblp.org/rec/conf/sigir/Shami0S25


In [63]:
# Remove square parentheses from column authors
dblp_results["authors"] = dblp_results["authors"].apply(
    lambda x: ", ".join(x) if isinstance(x, (list, tuple, np.ndarray)) else x
)

In [64]:
dblp_results.head(2)

,dblp_key,title,authors,year,venue,type,doi,url
20414,conf/sigir/Shami0S25,Fact Verification in Knowledge Graphs Using LLMs,"Farzad Shami, Stefano Marchesin 0001, Gianmari...",2025,SIGIR,Conference and Workshop Papers,10.1145/3726302.3730142,https://dblp.org/rec/conf/sigir/Shami0S25
22483,conf/sigir/DholeA26,RubricRAG: Towards Interpretable and Reliable ...,"Kaustubh D. Dhole, Eugene Agichtein",2026,SIGIR,Conference and Workshop Papers,10.1145/3805712.3809940,https://dblp.org/rec/conf/sigir/DholeA26


Most papers do not have abstract in CrossRef, so abstract retrieval step is skipped.

In [65]:
# def get_crossref_abstract(doi):
#     """Fetch the abstract for a given DOI from the Crossref API."""
#     if not doi:
#         return None
#     response = requests.get(f"https://api.crossref.org/works/{doi}", timeout=30)
#     if response.status_code != 200:
#         return None
#     abstract = response.json().get("message", {}).get("abstract")
#     if abstract:
#         return re.sub(r"<.*?>", "", abstract).strip()  # strip JATS/HTML tags
#     return None

# abstracts = []
# for doi in dblp_results["doi"]:
#     abstracts.append(get_crossref_abstract(doi))
#     time.sleep(0.1)  # be polite to the Crossref API

# dblp_results["abstract"] = abstracts
# print("With abstract:", dblp_results["abstract"].notna().sum(), "of", len(dblp_results))

In [66]:
# Rename columns for clarity
dblp_results.rename(columns={
    'title': 'Title',
    'authors': 'Authors',
    'year': 'Year',
    'venue': 'Venue',
    'type': 'Type',
    'url': 'URL',
    'doi': 'DOI',
    'dblp_key': 'DBLP Key'
}, inplace=True)

dblp_results.head(2)

,DBLP Key,Title,Authors,Year,Venue,Type,DOI,URL
20414,conf/sigir/Shami0S25,Fact Verification in Knowledge Graphs Using LLMs,"Farzad Shami, Stefano Marchesin 0001, Gianmari...",2025,SIGIR,Conference and Workshop Papers,10.1145/3726302.3730142,https://dblp.org/rec/conf/sigir/Shami0S25
22483,conf/sigir/DholeA26,RubricRAG: Towards Interpretable and Reliable ...,"Kaustubh D. Dhole, Eugene Agichtein",2026,SIGIR,Conference and Workshop Papers,10.1145/3805712.3809940,https://dblp.org/rec/conf/sigir/DholeA26


In [67]:
dblp_results.to_pickle("../results/dblp_results.pkl")

### ACL

Load external pickle file to search within a dataframe.

In [134]:
acl_df = pd.read_parquet("../data/acl_anthology.parquet")

In [135]:
acl_df.head(2)

,bib_key,ENTRYTYPE,title,abstract,author,booktitle,year,month,pages,url,publisher,anthology_id,venues,doi,language
0,wang-etal-2020-touch,inproceedings,Touch Editing: A Flexible One-Time Interaction...,We propose a touch-based editing method for tr...,"Wang, Qian andZhang, Jiajun andLiu, Lemao a...",Proceedings of the 1st Conference of the Asia-...,2020,December,1–11,https://aclanthology.org/2020.aacl-main.1/,Association for Computational Linguistics,2020.aacl-main.1,[aacl],10.18653/v1/2020.aacl-main.1,NaN
1,chi-etal-2020-monolingual,inproceedings,Can Monolingual Pretrained Models Help Cross-L...,Multilingual pretrained language models (such ...,"Chi, Zewen andDong, Li andWei, Furu andMao,...",Proceedings of the 1st Conference of the Asia-...,2020,December,12–17,https://aclanthology.org/2020.aacl-main.2/,Association for Computational Linguistics,2020.aacl-main.2,[aacl],10.18653/v1/2020.aacl-main.2,NaN


In [136]:
acl_df.year.unique()

<ArrowStringArray>
['2020', '2021', '2022', '2023', '2024', '2025', '2026']
Length: 7, dtype: str

In [137]:
acl_df.language.unique()

<ArrowStringArray>
[nan, 'eng', 'zho', 'fra', 'pt-BR', 'por', 'ita']
Length: 7, dtype: str

In [138]:
# Remove where language is not English
acl_df = acl_df[~acl_df['language'].isin(['fra', 'zho', 'ita', 'por', 'pt-BR'])]

In [139]:
# Remove columns that are not needed
acl_synthesis = acl_df.drop(columns=['ENTRYTYPE', 'pages', 'venues', 'language', 'month', 'anthology_id'])

In [140]:
# Rename columns for clarity
acl_synthesis.rename(columns={
    'title': 'Title',
    'author': 'Authors',
    'year': 'Year',
    'booktitle': 'Venue',
    'bib_key': 'ACL Key',
    'url': 'URL',
    'doi': 'DOI',
    'abstract': 'Abstract',
    'publisher': 'Publisher'
}, inplace=True)

In [141]:
acl_synthesis.head(2)

,ACL Key,Title,Abstract,Authors,Venue,Year,URL,Publisher,DOI
0,wang-etal-2020-touch,Touch Editing: A Flexible One-Time Interaction...,We propose a touch-based editing method for tr...,"Wang, Qian andZhang, Jiajun andLiu, Lemao a...",Proceedings of the 1st Conference of the Asia-...,2020,https://aclanthology.org/2020.aacl-main.1/,Association for Computational Linguistics,10.18653/v1/2020.aacl-main.1
1,chi-etal-2020-monolingual,Can Monolingual Pretrained Models Help Cross-L...,Multilingual pretrained language models (such ...,"Chi, Zewen andDong, Li andWei, Furu andMao,...",Proceedings of the 1st Conference of the Asia-...,2020,https://aclanthology.org/2020.aacl-main.2/,Association for Computational Linguistics,10.18653/v1/2020.aacl-main.2


In [142]:
# Count rows where abstract is NaN
len(acl_synthesis[acl_synthesis['Abstract'].isna()])

4406

In [143]:
def search_papers(df: pd.DataFrame, col: str = "Abstract") -> pd.DataFrame:
    """
    Search ACL papers by filtering abstracts for language models,
    factuality/hallucination, and knowledge graphs.
    """
    text = df[col]

    def has(keywords: str) -> pd.Series:
        return text.str.contains(f"(?:{keywords})", case=False, regex=True, na=False)

    mask = (
        has(acl_llm_keywords)
        & has(acl_kg_keywords)
        & has(acl_hallucination_keywords)
    )
    return df[mask]

In [144]:
acl_results = search_papers(acl_synthesis)

In [145]:
# Print number of results
print(f"Number of ACL papers matching criteria: {len(acl_results)}")

Number of ACL papers matching criteria: 139


In [146]:
acl_results.head(2)

,ACL Key,Title,Abstract,Authors,Venue,Year,URL,Publisher,DOI
9365,dziri-etal-2021-neural,Neural Path Hunter: Reducing Hallucination in ...,Dialogue systems powered by large pre-trained ...,"Dziri, Nouha andMadotto, Andrea andZaïane, O...",Proceedings of the 2021 Conference on Empirica...,2021,https://aclanthology.org/2021.emnlp-main.168/,Association for Computational Linguistics,10.18653/v1/2021.emnlp-main.168
14561,cao-etal-2022-hallucinated,Hallucinated but Factual! Inspecting the Factu...,State-of-the-art abstractive summarization sys...,"Cao, Meng andDong, Yue andCheung, Jackie",Proceedings of the 60th Annual Meeting of the ...,2022,https://aclanthology.org/2022.acl-long.236/,Association for Computational Linguistics,10.18653/v1/2022.acl-long.236


In [147]:
# Save to pickle
acl_results.to_pickle("../results/acl_results.pkl")

### ACM

Literature search performed through the website interface. Parsing takes as input the web interface export saved under `data`.

In [128]:
useful_fields = [
    "title",
    "author",
    "year",
    "booktitle",
    "journal",
    "publisher",
    "abstract",
    "doi",
    "url"
]

In [129]:
def load_bibtex(file_path: str) -> pd.DataFrame:
    with open(file_path, 'r') as bibtex_file:
        bib_database = bibtexparser.load(bibtex_file)

    print("Number of Papers:", len(bib_database.entries_dict.keys()))

    # Load the dictionary into a pandas dataframe
    df = pd.DataFrame.from_dict(bib_database.entries_dict, orient='index')

    return df

In [130]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    # Print information about years and languages
    print(df.year.unique())

    # Remove when year is < 2020
    df = df[df['year'].astype(int) >= 2020]

    # Keep only useful fields
    df = df[[col for col in useful_fields if col in df.columns]]

    # Rename columns for consistency
    renamed_df = df.rename(columns={
        'title': 'Title',
        'author': 'Authors',
        'year': 'Year',
        'booktitle': 'Venue',
        'journal': 'Journal',
        'url': 'URL',
        'doi': 'DOI',
        'abstract': 'Abstract',
        'publisher': 'Publisher'
    }).reset_index(drop=True)

    return renamed_df

In [131]:
acm_df = load_bibtex(ACM_FILE_NAME)
acm_df.head(2)

Number of Papers: 220


,keywords,numpages,month,journal,doi,url,issn,number,volume,address,...,ENTRYTYPE,ID,series,location,pages,booktitle,abstract,isbn,articleno,note
10.1016/j.neucom.2025.131230,"Knowledge graph, Large language model, Educati...",26,November,Neurocomput.,10.1016/j.neucom.2025.131230,https://doi.org/10.1016/j.neucom.2025.131230,0925-2312,C,654,NLD,...,article,10.1016/j.neucom.2025.131230,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10.1016/j.aei.2026.104327,"Urban rail transit signal system, Operation an...",22,June,Adv. Eng. Inform.,10.1016/j.aei.2026.104327,https://doi.org/10.1016/j.aei.2026.104327,1474-0346,PB,71,NLD,...,article,10.1016/j.aei.2026.104327,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [132]:
clean_acm_df = clean_dataframe(acm_df)
clean_acm_df.head(2)

<ArrowStringArray>
['2025', '2026', '2024', '2023']
Length: 4, dtype: str


,Title,Authors,Year,Venue,Journal,Publisher,Abstract,DOI,URL
0,Knowledge graph and large language model integ...,"Chen, Guanyu and Song, Tao and Wang, Quanyu an...",2025,NaN,Neurocomput.,Elsevier Science Publishers B. V.,NaN,10.1016/j.neucom.2025.131230,https://doi.org/10.1016/j.neucom.2025.131230
1,A unified LLM-KG framework for low‑annotation ...,"Cai, Wei and Zhu, Xiaomin and Sun, Zeyu and Ye...",2026,NaN,Adv. Eng. Inform.,Elsevier Science Publishers B. V.,NaN,10.1016/j.aei.2026.104327,https://doi.org/10.1016/j.aei.2026.104327


In [133]:
# Save to pickle
clean_acm_df.to_pickle("../results/acm_results.pkl")

### IEEE

Literature search performed through the website interface. Parsing takes as input the web interface export saved under `data`.

In [88]:
if IEEE_FILE_NAME.endswith(".bib"):
    ieee_df = load_bibtex(IEEE_FILE_NAME)
    ieee_df = clean_dataframe(ieee_df)
else:
    ieee_df = pd.read_csv(IEEE_FILE_NAME)
    # Keep useful fields and rename columns
    ieee_df = ieee_df.rename(columns={
        'Document Title': 'Title',
        'Authors': 'Authors',
        'Publication Year': 'Year',
        'Abstract': 'Abstract',
        'Publication Title': 'Venue',
        'Item DOI': 'DOI',
        'PDF Link': 'URL',
        'Document Identifier': 'Type',
        'Publisher': 'Publisher',
        'Article Citation Count': 'Cited By',
    })
    ieee_df = ieee_df[['Title', 'Authors', 'Year', 'Abstract', 'Venue', 'DOI', 'URL', 'Publisher', 'Type', 'Cited By']]

In [89]:
ieee_df.head(2)

,Title,Authors,Year,Abstract,Venue,DOI,URL,Publisher,Type,Cited By
0,Building Neo4j-Powered Applications with LLMs:...,R. Anthapu; S. Agarwal; D. J. Webber; D. J. Risch,2025,A comprehensive guide to building cutting-edge...,Building Neo4j-Powered Applications with LLMs:...,NaN,https://ieeexplore.ieee.org/xpl/ebooks/bookPdf...,Packt Publishing,Packt Publishing eBooks,NaN
1,"Building AI Agents with LLMs, RAG, and Knowled...",S. Raieli; G. Iuculano,2025,Master LLM fundamentals to advanced techniques...,"Building AI Agents with LLMs, RAG, and Knowled...",NaN,https://ieeexplore.ieee.org/xpl/ebooks/bookPdf...,Packt Publishing,Packt Publishing eBooks,NaN


In [90]:
# Save to pickle
ieee_df.to_pickle("../results/ieee_results.pkl")

### Springer

Literature search performed through the website interface. Parsing takes as input the web interface export saved under `data`.

In [91]:
springer_df = pd.read_csv(SPRINGER_FILE_NAME)
springer_df.head(2)

,Item Title,Publication Title,Book Series Title,Journal Volume,Journal Issue,Item DOI,Authors,Publication Year,URL,Content Type
0,Knowledge graph-enhanced large language models...,Scientific Reports,NaN,NaN,NaN,10.1038/s41598-026-69831-6,Xiaotong LiJunjie HuangQiang Zhang,2026,https://link.springer.com/article/10.1038/s415...,Article
1,Graph-Driven Thematic Review Summarization wit...,New Generation Computing,NaN,NaN,NaN,10.1007/s00354-026-00332-w,B. SuganyaM. L. Valarmathi,2026,https://link.springer.com/article/10.1007/s003...,Article


In [92]:
# Keep useful fields and rename columns
springer_df = springer_df.rename(columns={
    'Item Title': 'Title',
    'Publication Year': 'Year',
    'Publication Title': 'Venue',
    'Item DOI': 'DOI',
    'URL': 'URL',
    'Content Type': 'Type'
})
springer_df = springer_df[['Title', 'Authors', 'Year', 'Venue', 'DOI', 'URL', 'Type']]
springer_df.head(2)

,Title,Authors,Year,Venue,DOI,URL,Type
0,Knowledge graph-enhanced large language models...,Xiaotong LiJunjie HuangQiang Zhang,2026,Scientific Reports,10.1038/s41598-026-69831-6,https://link.springer.com/article/10.1038/s415...,Article
1,Graph-Driven Thematic Review Summarization wit...,B. SuganyaM. L. Valarmathi,2026,New Generation Computing,10.1007/s00354-026-00332-w,https://link.springer.com/article/10.1007/s003...,Article


In [93]:
print(springer_df['Year'].unique())
springer_df = springer_df[springer_df['Year'].astype(int) >= 2020].reset_index(drop=True)

[2026 2025 2024 2023]


In [94]:
# Save to pickle
springer_df.to_pickle("../results/springer_results.pkl")

### Scopus

In [121]:
SCOPUS_URL = "https://api.elsevier.com/content/search/scopus"
PAGE_SIZE = 25  # max per request

def scopus_search(query, max_results=200):
    """
    Search Scopus with boolean logic inside titles and optionally abstracts.
    Pages through results PAGE_SIZE at a time, up to max_results.
    Returns a list of dicts with the main fields.
    """
    headers = {"X-ELS-APIKey": SCOPUS_API_KEY, "Accept": "application/json"}
    results = []
    start = 0

    while start < max_results:
        params = {
            "query": query,
            "start": start,
            "count": min(PAGE_SIZE, max_results - start),
            "view": "STANDARD",
        }
        response = requests.get(SCOPUS_URL, headers=headers, params=params, timeout=30)
        if response.status_code != 200:
            print(response.text[:500])  # Elsevier's error message
        response.raise_for_status()

        data = response.json().get("search-results", {})
        entries = data.get("entry", [])

        # An empty result set comes back as a single {"error": "Result set was empty"} entry
        if not entries or "error" in entries[0]:
            break

        for e in entries:
            results.append({
                "title": e.get("dc:title"),
                "authors": e.get("dc:creator"),
                "doi": e.get("prism:doi"),
                "year": e.get("prism:coverDate", "")[:4],
                "scopus_id": e.get("dc:identifier"),
                "url": e.get("prism:url"),
                "citations": int(e.get("citedby-count", 0) or 0),
            })

        start += len(entries)
        total = int(data.get("opensearch:totalResults", 0))
        if start >= total:
            break

        time.sleep(0.2)  # stay under the 9 requests/second limit

    return results

Equivalent search on the website:

TITLE-ABS-KEY ( "language model*" OR llm* AND "knowledge graph*" OR kg* AND hallucinat* ) AND PUBYEAR > 2019 AND PUBYEAR < 2027 AND SUBJAREA ( COMP ) AND LANGUAGE ( english )

In [122]:
YEARS = range(2020, 2027)  # 2020 to 2026 inclusive
MAX_PER_YEAR = 500

yearly_results = []
for year in YEARS:
    query = (
        f"{scopus_query} "
        f"AND PUBYEAR > {year - 1} AND PUBYEAR < {year + 1} "
        "AND SUBJAREA ( COMP ) AND LANGUAGE ( english )"
    )
    try:
        results = scopus_search(query, max_results=MAX_PER_YEAR)
    except requests.HTTPError as e:
        print(f"{year}: request failed -> {e.response.status_code}")
        print(e.response.text[:500])  # Elsevier's explanation of the error
        continue

    year_df = results.copy() if isinstance(results, pd.DataFrame) else pd.DataFrame(results)
    print(f"{year}: {len(year_df)} results")
    yearly_results.append(year_df)

2020: 0 results
2021: 1 results
2022: 0 results
2023: 19 results
2024: 119 results
2025: 321 results
2026: 489 results


In [123]:
scopus_df = pd.concat(yearly_results, ignore_index=True)
print("Number of Scopus results 2020-2026:", len(scopus_df))

Number of Scopus results 2020-2026: 949


In [124]:
scopus_df.head(2)

,title,authors,doi,year,scopus_id,url,citations
0,Neural Path Hunter: Reducing Hallucination in ...,Dziri N.,10.18653/v1/2021.emnlp-main.168,2021,SCOPUS_ID:85127106950,https://api.elsevier.com/content/abstract/scop...,104
1,Knowledge Graphs Enhanced Large Language Model...,Wang C.,10.1145/3650400.3650405,2023,SCOPUS_ID:85191439580,https://api.elsevier.com/content/abstract/scop...,7


In [125]:
# Show how many papers have 0 or 1 citation
print("Papers with no citations:", len(scopus_df[scopus_df['citations'] == 0]))
print("Papers with 1 citation:", len(scopus_df[scopus_df['citations'] == 1]))

Papers with no citations: 519
Papers with 1 citation: 104


In [126]:
# Rename columns for clarity
scopus_df.rename(columns={
    'title': 'Title',
    'authors': 'Authors',
    'doi': 'DOI',
    'year': 'Year',
    'abstract': 'Abstract',
    'scopus_id': 'Scopus ID',
    'url': 'URL',
    'citations': 'Cited By'
}, inplace=True)

In [127]:
# Save to pickle
scopus_df.to_pickle("../results/scopus_results.pkl")

### arXiv

In [102]:
def arxiv_search(query, max_results=25):
    """
    Search arXiv by applying the query to both title and abstract.
    Boolean operators (AND, OR, ANDNOT) are supported.
    """

    with libreq.urlopen(f'http://export.arxiv.org/api/query?search_query={query}&start=0&max_results={max_results}') as url:
      data = url.read()

    feed = feedparser.parse(data)

    results = []
    for entry in feed.entries:
        results.append({
            "title": entry.title,
            "authors": [a.name for a in entry.authors],
            "summary": entry.summary,
            "published": entry.published,
            "updated": entry.updated,
            "pdf_url": next((l.href for l in entry.links if l.type == "application/pdf"), None),
            "arxiv_url": entry.link,
            "category": entry.arxiv_primary_category.get("term", "no")[:2]
        })

    return results

Alternative by title search:

'ti:("language model" OR llm OR encoder OR decoder OR gpt OR rag) AND ' \
'ti:(kg OR "knowledge graph" OR "knowledge base" OR "graph database") AND ' \
'ti:(hallucinat OR fact OR trust OR reliab OR explain)'

In [103]:
arxiv_query = 'abs:"language model" AND "knowledge graph" AND hallucination'

In [104]:
arxiv_results = arxiv_search(libreq.quote(arxiv_query), max_results=1000)

In [105]:
len(arxiv_results)

362

In [106]:
arxiv_titles = [entry['title'] for entry in arxiv_results]
arxiv_titles[:5]

['Enhancing Knowledge Graph Construction: Evaluating with Emphasis on Hallucination, Omission, and Graph Similarity Metrics',
 'Assessing LLMs Suitability for Knowledge Graph Completion',
 'CORE-Acu: Structured Reasoning Traces and Knowledge Graph Safety Verification for Acupuncture Clinical Decision Support',
 'Mitigating Large Language Model Hallucinations via Autonomous Knowledge Graph-based Retrofitting',
 'GraphEval: A Knowledge-Graph Based LLM Hallucination Evaluation Framework']

In [107]:
# Add resutls to a dataframe
arxiv_df = pd.DataFrame(arxiv_results)
arxiv_df.head(2)

,title,authors,summary,published,updated,pdf_url,arxiv_url,category
0,Enhancing Knowledge Graph Construction: Evalua...,"[Hussam Ghanem, Christophe Cruz]",Recent advancements in large language models h...,2025-02-07T11:19:01Z,2025-02-07T11:19:01Z,https://arxiv.org/pdf/2502.05239v1,https://arxiv.org/abs/2502.05239v1,cs
1,Assessing LLMs Suitability for Knowledge Graph...,"[Vasile Ionut Remus Iga, Gheorghe Cosmin Silaghi]",Recent work has shown the capability of Large ...,2024-05-27T15:04:50Z,2024-07-18T09:48:02Z,https://arxiv.org/pdf/2405.17249v2,https://arxiv.org/abs/2405.17249v2,cs


In [108]:
# Filter for category cs (computer science)
arxiv_cs_df = arxiv_df[arxiv_df['category'].str.startswith('cs')].reset_index(drop=True)
print(f"Number of arXiv CS papers matching criteria: {len(arxiv_cs_df)}")

Number of arXiv CS papers matching criteria: 360


In [109]:
# Extract year from updated date
arxiv_cs_df['updated_year'] = pd.to_datetime(arxiv_cs_df['updated']).dt.year

# Remove pdf_url column
arxiv_df_short = arxiv_cs_df[['title', 'authors', 'summary', 'updated_year', 'arxiv_url']]
arxiv_df_short.head(2)

,title,authors,summary,updated_year,arxiv_url
0,Enhancing Knowledge Graph Construction: Evalua...,"[Hussam Ghanem, Christophe Cruz]",Recent advancements in large language models h...,2025,https://arxiv.org/abs/2502.05239v1
1,Assessing LLMs Suitability for Knowledge Graph...,"[Vasile Ionut Remus Iga, Gheorghe Cosmin Silaghi]",Recent work has shown the capability of Large ...,2024,https://arxiv.org/abs/2405.17249v2


In [110]:
# Rename columns for clarity
arxiv_df_short = arxiv_df_short.rename(columns={
    'title': 'Title',
    'authors': 'Authors',
    'summary': 'Abstract',
    'updated_year': 'Year',
    'arxiv_url': 'URL'
})

In [111]:
# Select where year is 2020 or later
arxiv_df_2020_onwards = arxiv_df_short[arxiv_df_short['Year'] >= 2020].reset_index(drop=True)
arxiv_df_2020_onwards.head(2)

,Title,Authors,Abstract,Year,URL
0,Enhancing Knowledge Graph Construction: Evalua...,"[Hussam Ghanem, Christophe Cruz]",Recent advancements in large language models h...,2025,https://arxiv.org/abs/2502.05239v1
1,Assessing LLMs Suitability for Knowledge Graph...,"[Vasile Ionut Remus Iga, Gheorghe Cosmin Silaghi]",Recent work has shown the capability of Large ...,2024,https://arxiv.org/abs/2405.17249v2


In [112]:
len(arxiv_df_2020_onwards)

360

In [113]:
# Save to pickle
arxiv_df_2020_onwards.to_pickle("../results/arxiv_results.pkl")